In [1]:
import pandas as pd
import math
import networkx as nx
import os
import subprocess
import time
from SPARQLWrapper import SPARQLWrapper, JSON, CSV, N3, XML, TURTLE
import rdflib
import re
import IPython
import matplotlib.pyplot as plt
import numpy as np

In [2]:
endpoint_reactome = "http://localhost:3030/reactome"
rdfFormat = "turtle"
current_directory = os.getcwd()
BioPAX_Ontology_file_path = os.path.join(current_directory, '../', 'BioPAXData', 'biopax-level3.owl')
ReactomeBioPAX_file_path = os.path.join(current_directory, '../', 'BioPAXData', 'Homo_sapiens_v95.owl')

In [3]:
prefixes = f"""
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs:<http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX dc: <http://purl.org/dc/elements/1.1/>
PREFIX dcterms: <http://purl.org/dc/terms/>
PREFIX chebi: <http://purl.obolibrary.org/obo/chebi/>
PREFIX chebidb: <http://purl.obolibrary.org/obo/CHEBI_>
PREFIX chebirel: <http://purl.obolibrary.org/obo/CHEBI#>
PREFIX oboInOwl: <http://www.geneontology.org/formats/oboInOwl#>
PREFIX bp3: <http://www.biopax.org/release/biopax-level3.owl#>
PREFIX reactome: <http://www.reactome.org/biopax/40/48887#>
PREFIX abstraction:<http://abstraction/#>
"""

In [15]:
command = [
    '/home/cbeust/Softwares/JenaFuseki/apache-jena-fuseki-4.9.0/fuseki-server',
    '--file', ReactomeBioPAX_file_path,
    '--file', BioPAX_Ontology_file_path,
    '/reactome']

process = subprocess.Popen(command)
time.sleep(60)

11:28:37 INFO  Server          :: Dataset: in-memory: load file: /home/cbeust/Projects/2025/BioPAXPathwayAbstraction/Scripts/../BioPAXData/Homo_sapiens_v95.owl
11:28:39 WARN  riot            :: [line: 71253, col: 45] {W137} Input is large. Switching off checking for illegal reuse of rdf:ID's.
11:28:59 INFO  Server          :: Dataset: in-memory: load file: /home/cbeust/Projects/2025/BioPAXPathwayAbstraction/Scripts/../BioPAXData/biopax-level3.owl
11:28:59 INFO  Server          :: Running in read-only mode for /reactome
11:28:59 INFO  Server          :: Apache Jena Fuseki 4.9.0
11:28:59 INFO  Config          :: FUSEKI_HOME=/home/cbeust/Softwares/JenaFuseki/apache-jena-fuseki-4.9.0
11:28:59 INFO  Config          :: FUSEKI_BASE=/home/cbeust/Projects/2025/BioPAXPathwayAbstraction/Scripts/run
11:28:59 INFO  Config          :: Shiro file: file:///home/cbeust/Projects/2025/BioPAXPathwayAbstraction/Scripts/run/shiro.ini
11:29:00 INFO  Server          :: Database: in-memory, with files loaded
1

### 1.1 - Number of physical entities per pathway

In [5]:
query_pe_per_pathway = """ 
SELECT DISTINCT ?pathwayID ?entityToCount
WHERE {
  ?pathway rdf:type bp3:Pathway .
  ?pathway (bp3:pathwayComponent+|bp3:pathwayOrder/bp3:stepProcess) ?interaction .
  ?interaction rdf:type/(rdfs:subClassOf*) bp3:Interaction .
  
  {
    # Direct participants
    VALUES ?relation { bp3:left bp3:right bp3:participant }
    ?interaction ?relation ?entity .
    ?entity rdf:type/(rdfs:subClassOf*) bp3:PhysicalEntity .
  }
  UNION
  {
    # Controllers
    ?control rdf:type/(rdfs:subClassOf*) bp3:Control .
    ?control bp3:controlled ?interaction .
    ?control bp3:controller ?entity .
    ?entity rdf:type/(rdfs:subClassOf*) bp3:PhysicalEntity .
  }
  
  # For complex entities, get their components
  OPTIONAL {
    ?entity rdf:type bp3:Complex .
    ?entity (bp3:component+|bp3:memberPhysicalEntity+) ?entityCompo .
    BIND(?entityCompo AS ?entityToCount)
  }
  
  # For non-complex entities, use the entity itself
  OPTIONAL {
    FILTER NOT EXISTS { ?entity rdf:type bp3:Complex }
    BIND(?entity AS ?entityToCount)
  }
  
  ?pathway bp3:xref ?pathwayXref .
  ?pathwayXref rdf:type bp3:UnificationXref .
  ?pathwayXref bp3:db "Reactome" .
  ?pathwayXref bp3:id ?pathwayID .
}
"""

# execute SPARQL query
sparql = SPARQLWrapper(endpoint_reactome)
sparql.setQuery(prefixes+query_pe_per_pathway)
# export results to CSV
sparql.setReturnFormat(CSV)
results = sparql.query().convert()
with open(f"../Results/Reactome95/ReactomeHomoSapiens95UtilityFiles/ReactomeHomoSapiens95_PePerPathway.csv", "wb") as f:
    f.write(results)

09:57:05 INFO  Fuseki          :: [1] GET http://localhost:3030/reactome?query=%0APREFIX+rdf%3A+%3Chttp%3A//www.w3.org/1999/02/22-rdf-syntax-ns%23%3E%0APREFIX+rdfs%3A%3Chttp%3A//www.w3.org/2000/01/rdf-schema%23%3E%0APREFIX+owl%3A+%3Chttp%3A//www.w3.org/2002/07/owl%23%3E%0APREFIX+xsd%3A+%3Chttp%3A//www.w3.org/2001/XMLSchema%23%3E%0APREFIX+dc%3A+%3Chttp%3A//purl.org/dc/elements/1.1/%3E%0APREFIX+dcterms%3A+%3Chttp%3A//purl.org/dc/terms/%3E%0APREFIX+chebi%3A+%3Chttp%3A//purl.obolibrary.org/obo/chebi/%3E%0APREFIX+chebidb%3A+%3Chttp%3A//purl.obolibrary.org/obo/CHEBI_%3E%0APREFIX+chebirel%3A+%3Chttp%3A//purl.obolibrary.org/obo/CHEBI%23%3E%0APREFIX+oboInOwl%3A+%3Chttp%3A//www.geneontology.org/formats/oboInOwl%23%3E%0APREFIX+bp3%3A+%3Chttp%3A//www.biopax.org/release/biopax-level3.owl%23%3E%0APREFIX+reactome%3A+%3Chttp%3A//www.reactome.org/biopax/40/48887%23%3E%0APREFIX+abstraction%3A%3Chttp%3A//abstraction/%23%3E%0A+%0ASELECT+DISTINCT+%3FpathwayID+%3FentityToCount%0AWHERE+%7B%0A++%3Fpathway+rdf

In [6]:
# replace URIs in the file
df = pd.read_csv("../Results/Reactome95/ReactomeHomoSapiens95UtilityFiles/ReactomeHomoSapiens95_PePerPathway.csv", sep=",", header=None)
df = df.replace('http://www.reactome.org/biopax/95/48887#', 'reactome:', regex=True)
print(df.head())
df.to_csv("../Results/Reactome95/ReactomeHomoSapiens95UtilityFiles/ReactomeHomoSapiens95_PePerPathway.csv", sep=",", header=None, index=False)

              0                           1
0     pathwayID               entityToCount
1  R-HSA-140875  reactome:PhysicalEntity918
2  R-HSA-140875        reactome:Complex4197
3  R-HSA-140875       reactome:Protein11709
4  R-HSA-140875       reactome:Protein11708


### 1.2 - Number of Physical Entities per pathway (without complexes)

In [7]:
query_pe_per_pathway_without_complexes = """ 
SELECT DISTINCT ?pathwayID ?entity
WHERE {
  ?pathway rdf:type bp3:Pathway .
  ?pathway (bp3:pathwayComponent+|bp3:pathwayOrder/bp3:stepProcess) ?interaction .
  ?interaction rdf:type/(rdfs:subClassOf*) bp3:Interaction .
  
  {
    # Direct participants
    VALUES ?relation { bp3:left bp3:right bp3:participant }
    ?interaction ?relation ?entity .
    ?entity rdf:type/(rdfs:subClassOf*) bp3:PhysicalEntity .
  }
  UNION
  {
    # Controllers
    ?control rdf:type/(rdfs:subClassOf*) bp3:Control .
    ?control bp3:controlled ?interaction .
    ?control bp3:controller ?entity .
    ?entity rdf:type/(rdfs:subClassOf*) bp3:PhysicalEntity .
  }
  
  ?pathway bp3:xref ?pathwayXref .
  ?pathwayXref rdf:type bp3:UnificationXref .
  ?pathwayXref bp3:db "Reactome" .
  ?pathwayXref bp3:id ?pathwayID .

  FILTER NOT EXISTS { ?entity rdf:type bp3:Complex . }
}
"""

# execute SPARQL query
sparql = SPARQLWrapper(endpoint_reactome)
sparql.setQuery(prefixes+query_pe_per_pathway_without_complexes)
# export results to CSV
sparql.setReturnFormat(CSV)
results = sparql.query().convert()
with open(f"../Results/Reactome95/ReactomeHomoSapiens95UtilityFiles/ReactomeHomoSapiens95_PePerPathwayWithoutComplexes.csv", "wb") as f:
    f.write(results)

10:16:42 INFO  Fuseki          :: [2] GET http://localhost:3030/reactome?query=%0APREFIX+rdf%3A+%3Chttp%3A//www.w3.org/1999/02/22-rdf-syntax-ns%23%3E%0APREFIX+rdfs%3A%3Chttp%3A//www.w3.org/2000/01/rdf-schema%23%3E%0APREFIX+owl%3A+%3Chttp%3A//www.w3.org/2002/07/owl%23%3E%0APREFIX+xsd%3A+%3Chttp%3A//www.w3.org/2001/XMLSchema%23%3E%0APREFIX+dc%3A+%3Chttp%3A//purl.org/dc/elements/1.1/%3E%0APREFIX+dcterms%3A+%3Chttp%3A//purl.org/dc/terms/%3E%0APREFIX+chebi%3A+%3Chttp%3A//purl.obolibrary.org/obo/chebi/%3E%0APREFIX+chebidb%3A+%3Chttp%3A//purl.obolibrary.org/obo/CHEBI_%3E%0APREFIX+chebirel%3A+%3Chttp%3A//purl.obolibrary.org/obo/CHEBI%23%3E%0APREFIX+oboInOwl%3A+%3Chttp%3A//www.geneontology.org/formats/oboInOwl%23%3E%0APREFIX+bp3%3A+%3Chttp%3A//www.biopax.org/release/biopax-level3.owl%23%3E%0APREFIX+reactome%3A+%3Chttp%3A//www.reactome.org/biopax/40/48887%23%3E%0APREFIX+abstraction%3A%3Chttp%3A//abstraction/%23%3E%0A+%0ASELECT+DISTINCT+%3FpathwayID+%3Fentity%0AWHERE+%7B%0A++%3Fpathway+rdf%3Atype

In [8]:
# replace URIs in the file
df = pd.read_csv("../Results/Reactome95/ReactomeHomoSapiens95UtilityFiles/ReactomeHomoSapiens95_PePerPathwayWithoutComplexes.csv", sep=",", header=None)
df = df.replace('http://www.reactome.org/biopax/95/48887#', 'reactome:', regex=True)
print(df.head())
df.to_csv("../Results/Reactome95/ReactomeHomoSapiens95UtilityFiles/ReactomeHomoSapiens95_PePerPathwayWithoutComplexes.csv", sep=",", header=None, index=False)

              0                           1
0     pathwayID                      entity
1  R-HSA-140875  reactome:PhysicalEntity918
2  R-HSA-140875   reactome:SmallMolecule110
3  R-HSA-140875       reactome:Protein11703
4  R-HSA-140875       reactome:Protein11694


### 1.3 - Number of EntityReferences per pathway

In [9]:
query_er_per_pathway = """ 
SELECT DISTINCT ?pathwayID ?entityRef
WHERE {
  VALUES ?db { "ChEBI" "UniProt" }
  ?pathway rdf:type bp3:Pathway .
  ?pathway (bp3:pathwayComponent+|bp3:pathwayOrder/bp3:stepProcess) ?interaction .
  ?interaction rdf:type/(rdfs:subClassOf*) bp3:Interaction .
  ?pathway bp3:xref ?pathwayXref .
  ?pathwayXref rdf:type bp3:UnificationXref .
  ?pathwayXref bp3:db "Reactome" .
  ?pathwayXref bp3:id ?pathwayID .
  
  {
    # Direct participants
    VALUES ?relation { bp3:left bp3:right bp3:participant }
    ?interaction ?relation ?entity .
    ?entity rdf:type/(rdfs:subClassOf*) bp3:PhysicalEntity .
  }
  UNION
  {
    # Controllers
    ?control rdf:type/(rdfs:subClassOf*) bp3:Control .
    ?control bp3:controlled ?interaction .
    ?control bp3:controller ?entity .
    ?entity rdf:type/(rdfs:subClassOf*) bp3:PhysicalEntity .
  }
  
  # For complex entities, get their components
  OPTIONAL {
    ?entity rdf:type bp3:Complex .
    ?entity (bp3:component+|bp3:memberPhysicalEntity+) ?entityCompo .
    ?entityCompo bp3:entityReference ?entityRef .
    ?entityRef bp3:xref ?entityRefXref .
    ?entityRefXref rdf:type bp3:UnificationXref .
    #?entityRefXref bp3:db ?db .
    #?entityRefXref bp3:id ?entityID .
  }
  
  # For non-complex entities, use the entity itself
  OPTIONAL {
    FILTER NOT EXISTS { ?entity rdf:type bp3:Complex . }
    ?entity bp3:entityReference ?entityRef .
    ?entityRef bp3:xref ?entityRefXref .
    ?entityRefXref rdf:type bp3:UnificationXref .
    #?entityRefXref bp3:db ?db .
    #?entityRefXref bp3:id ?entityID .
  }
}
"""

# execute SPARQL query
sparql = SPARQLWrapper(endpoint_reactome)
sparql.setQuery(prefixes+query_er_per_pathway)
# export results to CSV
sparql.setReturnFormat(CSV)
results = sparql.query().convert()
with open(f"../Results/Reactome95/ReactomeHomoSapiens95UtilityFiles/ReactomeHomoSapiens95_ErPerPathway.csv", "wb") as f:
    f.write(results)

10:49:05 INFO  Fuseki          :: [3] GET http://localhost:3030/reactome?query=%0APREFIX+rdf%3A+%3Chttp%3A//www.w3.org/1999/02/22-rdf-syntax-ns%23%3E%0APREFIX+rdfs%3A%3Chttp%3A//www.w3.org/2000/01/rdf-schema%23%3E%0APREFIX+owl%3A+%3Chttp%3A//www.w3.org/2002/07/owl%23%3E%0APREFIX+xsd%3A+%3Chttp%3A//www.w3.org/2001/XMLSchema%23%3E%0APREFIX+dc%3A+%3Chttp%3A//purl.org/dc/elements/1.1/%3E%0APREFIX+dcterms%3A+%3Chttp%3A//purl.org/dc/terms/%3E%0APREFIX+chebi%3A+%3Chttp%3A//purl.obolibrary.org/obo/chebi/%3E%0APREFIX+chebidb%3A+%3Chttp%3A//purl.obolibrary.org/obo/CHEBI_%3E%0APREFIX+chebirel%3A+%3Chttp%3A//purl.obolibrary.org/obo/CHEBI%23%3E%0APREFIX+oboInOwl%3A+%3Chttp%3A//www.geneontology.org/formats/oboInOwl%23%3E%0APREFIX+bp3%3A+%3Chttp%3A//www.biopax.org/release/biopax-level3.owl%23%3E%0APREFIX+reactome%3A+%3Chttp%3A//www.reactome.org/biopax/40/48887%23%3E%0APREFIX+abstraction%3A%3Chttp%3A//abstraction/%23%3E%0A+%0ASELECT+DISTINCT+%3FpathwayID+%3FentityRef%0AWHERE+%7B%0A++VALUES+%3Fdb+%7B+%

In [10]:
# replace URIs in the file
df = pd.read_csv("../Results/Reactome95/ReactomeHomoSapiens95UtilityFiles/ReactomeHomoSapiens95_ErPerPathway.csv", sep=",", header=None)
df = df.replace('http://www.reactome.org/biopax/95/48887#', 'reactome:', regex=True)
print(df.head())
df.to_csv("../Results/Reactome95/ReactomeHomoSapiens95UtilityFiles/ReactomeHomoSapiens95_ErPerPathway.csv", sep=",", header=None, index=False)

              0                                  1
0     pathwayID                          entityRef
1  R-HSA-140875      reactome:ProteinReference5097
2  R-HSA-140875  reactome:SmallMoleculeReference46
3  R-HSA-140875      reactome:ProteinReference3926
4  R-HSA-140875                                NaN


### 1.4 - UniProt IDs per pathway

In [16]:
query_uniprot_per_pathway = """ 
SELECT DISTINCT ?pathwayID ?entityRef ?entityID
WHERE {
  VALUES ?db { "UniProt" }
  ?pathway rdf:type bp3:Pathway .
  ?pathway (bp3:pathwayComponent+|bp3:pathwayOrder/bp3:stepProcess) ?interaction .
  ?interaction rdf:type/(rdfs:subClassOf*) bp3:Interaction .
  ?pathway bp3:xref ?pathwayXref .
  ?pathwayXref rdf:type bp3:UnificationXref .
  ?pathwayXref bp3:db "Reactome" .
  ?pathwayXref bp3:id ?pathwayID .
  
  {
    # Direct participants
    VALUES ?relation { bp3:left bp3:right bp3:participant }
    ?interaction ?relation ?entity .
    ?entity rdf:type/(rdfs:subClassOf*) bp3:PhysicalEntity .
  }
  UNION
  {
    # Controllers
    ?control rdf:type/(rdfs:subClassOf*) bp3:Control .
    ?control bp3:controlled ?interaction .
    ?control bp3:controller ?entity .
    ?entity rdf:type/(rdfs:subClassOf*) bp3:PhysicalEntity .
  }
  
  # For complex entities, get their components
  OPTIONAL {
    ?entity rdf:type bp3:Complex .
    ?entity (bp3:component+|bp3:memberPhysicalEntity+) ?entityCompo .
    ?entityCompo bp3:entityReference ?entityRef .
    ?entityRef bp3:xref ?entityRefXref .
    ?entityRefXref rdf:type bp3:UnificationXref .
    ?entityRefXref bp3:db ?db .
    ?entityRefXref bp3:id ?entityID .
  }
  
  # For non-complex entities, use the entity itself
  OPTIONAL {
    FILTER NOT EXISTS { ?entity rdf:type bp3:Complex . }
    ?entity bp3:entityReference ?entityRef .
    ?entityRef bp3:xref ?entityRefXref .
    ?entityRefXref rdf:type bp3:UnificationXref .
    ?entityRefXref bp3:db ?db .
    ?entityRefXref bp3:id ?entityID .
  }
}
"""

# execute SPARQL query
sparql = SPARQLWrapper(endpoint_reactome)
sparql.setQuery(prefixes+query_uniprot_per_pathway)
# export results to CSV
sparql.setReturnFormat(CSV)
results = sparql.query().convert()
with open(f"../Results/Reactome95/ReactomeHomoSapiens95UtilityFiles/ReactomeHomoSapiens95_UpIdPerPathway.csv", "wb") as f:
    f.write(results)

11:30:24 INFO  Fuseki          :: [1] GET http://localhost:3030/reactome?query=%0APREFIX+rdf%3A+%3Chttp%3A//www.w3.org/1999/02/22-rdf-syntax-ns%23%3E%0APREFIX+rdfs%3A%3Chttp%3A//www.w3.org/2000/01/rdf-schema%23%3E%0APREFIX+owl%3A+%3Chttp%3A//www.w3.org/2002/07/owl%23%3E%0APREFIX+xsd%3A+%3Chttp%3A//www.w3.org/2001/XMLSchema%23%3E%0APREFIX+dc%3A+%3Chttp%3A//purl.org/dc/elements/1.1/%3E%0APREFIX+dcterms%3A+%3Chttp%3A//purl.org/dc/terms/%3E%0APREFIX+chebi%3A+%3Chttp%3A//purl.obolibrary.org/obo/chebi/%3E%0APREFIX+chebidb%3A+%3Chttp%3A//purl.obolibrary.org/obo/CHEBI_%3E%0APREFIX+chebirel%3A+%3Chttp%3A//purl.obolibrary.org/obo/CHEBI%23%3E%0APREFIX+oboInOwl%3A+%3Chttp%3A//www.geneontology.org/formats/oboInOwl%23%3E%0APREFIX+bp3%3A+%3Chttp%3A//www.biopax.org/release/biopax-level3.owl%23%3E%0APREFIX+reactome%3A+%3Chttp%3A//www.reactome.org/biopax/40/48887%23%3E%0APREFIX+abstraction%3A%3Chttp%3A//abstraction/%23%3E%0A+%0ASELECT+DISTINCT+%3FpathwayID+%3FentityRef+%3FentityID%0AWHERE+%7B%0A++VALUES

KeyboardInterrupt: 

In [ ]:
query_uniprot_per_pathway = """ 
SELECT DISTINCT ?pathwayID ?entityID
WHERE {
  VALUES ?db { "UniProt" }
  ?pathway rdf:type bp3:Pathway .
  ?pathway (bp3:pathwayComponent+|bp3:pathwayOrder/bp3:stepProcess) ?interaction .
  ?interaction rdf:type/(rdfs:subClassOf*) bp3:Interaction .
  ?pathway bp3:xref ?pathwayXref .
  ?pathwayXref rdf:type bp3:UnificationXref .
  ?pathwayXref bp3:db "Reactome" .
  ?pathwayXref bp3:id ?pathwayID .
  
  {
    # Direct participants
    VALUES ?relation { bp3:left bp3:right bp3:participant }
    ?interaction ?relation ?entity .
    ?entity rdf:type/(rdfs:subClassOf*) bp3:PhysicalEntity .
  }
  UNION
  {
    # Controllers
    ?control rdf:type/(rdfs:subClassOf*) bp3:Control .
    ?control bp3:controlled ?interaction .
    ?control bp3:controller ?entity .
    ?entity rdf:type/(rdfs:subClassOf*) bp3:PhysicalEntity .
  }
  
  # For complex entities, get their components
  OPTIONAL {
    ?entity rdf:type bp3:Complex .
    ?entity (bp3:component+|bp3:memberPhysicalEntity+) ?entityCompo .
    ?entityCompo bp3:entityReference ?entityRef .
    ?entityRef bp3:xref ?entityRefXref .
    ?entityRefXref rdf:type bp3:UnificationXref .
    ?entityRefXref bp3:db ?db .
    ?entityRefXref bp3:id ?entityID .
  }

  OPTIONAL {
    ?entity rdf:type bp3:Protein .
    ?entity bp3:memberPhysicalEntity+ ?entityCompoProt .
    ?entityCompoProt bp3:entityReference ?entityCompoProtRef .
    ?entityCompoProtRef bp3:xref ?entityCompoProtRefXref .
    ?entityCompoProtRefXref rdf:type bp3:UnificationXref .
    ?entityCompoProtRefXref bp3:db ?db .
    ?entityCompoProtRefXref bp3:id ?entityID .
  }
  
  # For non-complex entities, use the entity itself
  OPTIONAL {
    FILTER NOT EXISTS { ?entity rdf:type bp3:Complex . }
    ?entity bp3:entityReference ?entityRef .
    ?entityRef bp3:xref ?entityRefXref .
    ?entityRefXref rdf:type bp3:UnificationXref .
    ?entityRefXref bp3:db ?db .
    ?entityRefXref bp3:id ?entityID .
  }
}
"""

# execute SPARQL query
sparql = SPARQLWrapper(endpoint_reactome)
sparql.setQuery(prefixes+query_uniprot_per_pathway)
# export results to CSV
sparql.setReturnFormat(CSV)
results = sparql.query().convert()
with open(f"../Results/Reactome95/ReactomeHomoSapiens95UtilityFiles/ReactomeHomoSapiens95_UpIdPerPathwayV2.csv", "wb") as f:
    f.write(results)

13:17:25 INFO  Fuseki          :: [6] GET http://localhost:3030/reactome?query=%0APREFIX+rdf%3A+%3Chttp%3A//www.w3.org/1999/02/22-rdf-syntax-ns%23%3E%0APREFIX+rdfs%3A%3Chttp%3A//www.w3.org/2000/01/rdf-schema%23%3E%0APREFIX+owl%3A+%3Chttp%3A//www.w3.org/2002/07/owl%23%3E%0APREFIX+xsd%3A+%3Chttp%3A//www.w3.org/2001/XMLSchema%23%3E%0APREFIX+dc%3A+%3Chttp%3A//purl.org/dc/elements/1.1/%3E%0APREFIX+dcterms%3A+%3Chttp%3A//purl.org/dc/terms/%3E%0APREFIX+chebi%3A+%3Chttp%3A//purl.obolibrary.org/obo/chebi/%3E%0APREFIX+chebidb%3A+%3Chttp%3A//purl.obolibrary.org/obo/CHEBI_%3E%0APREFIX+chebirel%3A+%3Chttp%3A//purl.obolibrary.org/obo/CHEBI%23%3E%0APREFIX+oboInOwl%3A+%3Chttp%3A//www.geneontology.org/formats/oboInOwl%23%3E%0APREFIX+bp3%3A+%3Chttp%3A//www.biopax.org/release/biopax-level3.owl%23%3E%0APREFIX+reactome%3A+%3Chttp%3A//www.reactome.org/biopax/40/48887%23%3E%0APREFIX+abstraction%3A%3Chttp%3A//abstraction/%23%3E%0A+%0ASELECT+DISTINCT+%3FpathwayID+%3FentityID%0AWHERE+%7B%0A++VALUES+%3Fdb+%7B+%2

In [17]:
process.kill()
time.sleep(60)